In [37]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteMLE
from pgmpy.inference import VariableElimination

In [39]:
# Load Heart Disease dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

cols = ["age", "sex", "cp", "trestbps", "chol", "fbs",
        "restecg", "thalach", "exang", "oldpeak", "slope",
        "ca", "thal", "target"]


In [41]:
data = pd.read_csv(url, names=cols, na_values="?")

In [42]:
data.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [45]:
# Select required medical attributes
data = data[["sex", "cp", "exang", "restecg", "target"]].dropna()

In [47]:
# Convert diagnosis into binary values
data["target"] = (data["target"] > 0).astype(int)

In [15]:

# Construct Bayesian Network
model = DiscreteBayesianNetwork([
    ("sex", "target"),
    ("cp", "target"),
    ("exang", "target"),
    ("restecg", "target")
])

In [51]:
# Learn probabilities
model.fit(data, estimator=DiscreteMLE())

In [53]:
# Perform inference
infer = VariableElimination(model)

In [55]:
result = infer.query(
    variables=["target"],
    evidence={"sex": 1, "cp": 4, "exang": 1, "restecg": 2}
)

print(result)

+-----------+---------------+
| target    |   phi(target) |
+===========+===============+
| target(0) |        0.0789 |
+-----------+---------------+
| target(1) |        0.9211 |
+-----------+---------------+


In [59]:
states = result.state_names["target"]
probabilities = result.values

p_no_disease = probabilities[states.index(0)]
p_disease = probabilities[states.index(1)]

In [61]:
print(f"Probability of no heart disease: {p_no_disease * 100:.2f}%")
print(f"Probability of heart disease:    {p_disease * 100:.2f}%")

Probability of no heart disease: 7.89%
Probability of heart disease:    92.11%
